In [2]:
import pandas as pd

file_path="penguins_size.csv"

#To read csv file
df=pd.read_csv(file_path)

Exception ignored in PyObject_HasAttr(); consider using PyObject_HasAttrWithError(), PyObject_GetOptionalAttr() or PyObject_GetAttr():
Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
AttributeError: partially initialized module 'pandas' from 'c:\Users\USER\Desktop\python\.venv\Lib\site-packages\pandas\__init__.py' has no attribute '_pandas_datetime_CAPI' (most likely due to a circular import)
Exception ignored in PyObject_HasAttr(); consider using PyObject_HasAttrWithError(), PyObject_GetOptionalAttr() or PyObject_GetAttr():
Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
AttributeError: partially initialized module 'pandas' from 'c:\Users\USER\Desktop\python\.venv\Lib\site-packages\pandas\__init__.py' has no attribute '_pandas_parser_CAPI' (most likely due to a circular import)


: 

In [ ]:
df.head

In [ ]:
from sklearn.compose import ColumnTransformer #triggers columns and changes the value
from sklearn.impute import SimpleImputer 
# fills the missing using startegies. default= mean value fills missing values, median, mode , most frequent, constant value

#applies transformer to columns returns numpy array, pandas df
trf1=ColumnTransformer([('impute_numerical',SimpleImputer(strategy="mean"),["culmen_length_mm","culmen_depth_mm","flipper_length_mm","body_mass_g"])],remainder="passthrough", verbose_feature_names_out=False).set_output(transform="pandas") #takes list inside tuple
 

In [ ]:
trf2=ColumnTransformer([('impute_gender',SimpleImputer(strategy="most_frequent"),["sex"])],remainder="passthrough", verbose_feature_names_out=False).set_output(transform="pandas") #takes list inside tuple


In [ ]:
#Add New feature
# transform - column ma implement garxa
# fit - takes the 
from sklearn.base import BaseEstimator,TransformerMixin

class CulmenRatio(BaseEstimator, TransformerMixin):
    def fit(self,x,y=None):
        return self
    
    def transform(self,x):
        x["culmen_ratio"]=x["culmen_length_mm"]/x["culmen_depth_mm"]
        return x

In [ ]:
from sklearn.preprocessing import OneHotEncoder

#sparse utput - matrix sparse ma dinxa
trf3=ColumnTransformer([("ohe",OneHotEncoder(sparse_output=False),["species","island","sex"])],remainder="passthrough", verbose_feature_names_out=False).set_output(transform="pandas")

In [ ]:
from sklearn.preprocessing import MinMaxScaler


trf4=ColumnTransformer([("scaling",MinMaxScaler(),["culmen_length_mm","culmen_depth_mm","flipper_length_mm"])],remainder="passthrough", verbose_feature_names_out=False).set_output(transform="pandas")

In [ ]:
"""Pipeline for integration"""

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("trans1", trf1),
    ("trans2", trf2),
    ("culmenratio", CulmenRatio()),
    ("trans3", trf3),
    ("trans4", trf4)
])

KeyboardInterrupt: 

In [ ]:
pipe

ValueError: too many values to unpack (expected 2)

ValueError: too many values to unpack (expected 2)

Pipeline(steps=[(('trans1',
                  ColumnTransformer(remainder='passthrough',
                                    transformers=[('impute_numerical',
                                                   SimpleImputer(),
                                                   ['culmen_length_mm',
                                                    'culmen_depth_mm',
                                                    'flipper_length_mm',
                                                    'body_mass_g'])],
                                    verbose_feature_names_out=False)),
                 ('trans2',
                  ColumnTransformer(remainder='passthrough',
                                    transformers=[('impute_gender',
                                                   SimpleImputer(strategy='most_frequent')...
                  ColumnTransformer(remainder='passthrough',
                                    transformers=[('ohe',
                                             

In [ ]:
df.head(10)

In [ ]:
import numpy as np

df["sex"]=df["sex"].replace(".",np.nan)

In [ ]:
df_transformed=pipe

In [ ]:
df_transformed

In [ ]:
from sklearn.base import BaseEstimator,TransformerMixin

class GenderImputer(BaseEstimator,TransformerMixin):
    def fit(self,x,y=None):
        self.mask=x[x["sex"].isin(['Male','Female'])]
        self.mass_median=self.mask.groupby(["species","sex"])["body_mass_g"].median()
        return self
    
    def transform(self,x):
        def fill_empty(row):
            spec = row["species"]
            male_mass = self.mass_median[spec]["MALE"]
            female_mass = self.mass_median[spec]["FEMALE"]

            threshold = (male_mass + female_mass) / 2
            if  row["body_mass_g"] >= threshold:
                return "MALE"
            else:
                return "FEMALE"
        
        self.mask_gender = (x["sex"].isna())
        self.mask_gender.sum()
        
        x.loc[self.mask_gender,"sex"] = x[self.mask_gender].apply(fill_empty,axis = 1)
        return x[["species","sex","body_mass_g"]]
    
    
    def get_feature_names_out(self,input_features=None):
        return np.array(["species","sex","body_mass_g"],dtype=object)


In [ ]:
trf2=ColumnTransformer([('impute_gender',GenderImputer(),["sex"])],remainder="passthrough", verbose_feature_names_out=False).set_output(transform="pandas") #takes list inside tuple


In [ ]:
# Calculate median of body mass -  only if sex is defined
